In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
import re
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold
import gc
from tqdm.notebook import tqdm

import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import Ridge, Lasso

# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Dense, Flatten, Conv1D, MaxPooling1D, concatenate
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import EarlyStopping

tqdm.pandas()

# For reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

In [ ]:
class Config:
    N_SPLITS = 5
    
    # OUTPUT_DIR = os.getcwd()
    TRAIN = "/kaggle/input/amlc2025-dataset/train.csv"
    TEST = "/kaggle/input/amlc2025-dataset/test.csv"
    TEXT_TRAIN = "/kaggle/input/amlc2025-dataset/combined_embeddings.npy"
    TEXT_TEST = "/kaggle/input/amlc2025-dataset/test_combined_embeddings.npy"
    IMAGE_TRAIN = "/kaggle/input/amlc-25/train_image_embeds.npy"
    IMAGE_TEST = "/kaggle/input/amlc-25/test_image_embeds.npy"
    TRAIN_SAMPLES = "/kaggle/working/train_samples.npy"
    TEST_SAMPLES = "/kaggle/working/test_samples.npy"
    
    N_SPLITS = 3
    SEED = 42
    
    # ANN Config
    ANN_EPOCHS = 25
    ANN_BATCH_SIZE = 128
    ANN_LEARNING_RATE = 1e-3

config = Config()

In [ ]:
def smape(y_true, y_pred):
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(numerator / denominator) * 100

In [1]:
# Few training samples were dropped and one testing samples was less
# These are the ones that were in the dataset
train_samples = np.load(config.TRAIN_SAMPLES)
test_samples = np.load(config.TEST_SAMPLES)

NameError: name 'np' is not defined

In [ ]:
print("1. Loading all data and pre-computed features...")
train_df_original = pd.read_csv(config.TRAIN)
test_df = pd.read_csv(config.TEST)

train_remove_indices = np.where(~train_df_original["sample_id"].isin(train_samples))[0]
test_remove_indices = np.where(~test_df["sample_id"].isin(test_samples))[0]

train_df = train_df_original.drop(train_remove_indices)
print(f"Train DF shape: {train_df_original.shape} reduced to {train_df.shape}, Test DF shape: {test_df.shape}")

print("   - Loading text embeddings...")
train_text_embeds_original = np.load(config.TEXT_TRAIN)
test_text_embeds = np.load(config.TEXT_TEST)

train_text_embeds = np.delete(train_text_embeds_original, train_remove_indices, axis = 0)
print(f"Train text embeds shape: {train_text_embeds_original.shape} reduced to {train_text_embeds.shape}, Test text embeds shape: {test_text_embeds.shape}")

print("   - Loading image embeddings...")
train_image_embeds = np.load(config.IMAGE_TRAIN)
test_image_embeds_incomplete = np.load(config.IMAGE_TEST)

test_image_embeds = np.insert(test_image_embeds_incomplete, test_remove_indices, np.mean(test_image_embeds_incomplete, axis = 0), axis = 0)
print(f"Train image embeds shape: {train_image_embeds.shape}, Test image embeds shape: {test_image_embeds_incomplete.shape} increased to {test_image_embeds.shape}")

1. Loading all data and pre-computed features...
Train DF shape: (75000, 4) reduced to (74972, 4), Test DF shape: (75000, 3)
   - Loading text embeddings...
Train text embeds shape: (75000, 2688) reduced to (74972, 2688), Test text embeds shape: (75000, 2688)
   - Loading image embeddings...
Train image embeds shape: (74972, 768), Test image embeds shape: (74999, 768) increased to (75000, 768)


In [ ]:
def extract_features(text: str) -> dict:
    if not isinstance(text, str):
        text = ""  # Handle potential NaN values

    features = {}
    lower_text = text.lower()

    # Pack Count (IPQ)
    pack_match = re.search(r'(?:pack of|set of|pk of|pack|pk)\s*(\d+)|(\d+)\s*(?:pack|pk|count|ct)', text, flags=re.I)
    features['pack_count'] = int(pack_match.group(1) or pack_match.group(2)) if pack_match else 1

    # Numeric Quantity and Unit (e.g., "16.5 oz", "2 lbs")
    qty_match = re.search(r'(\d+(?:\.\d+)?)\s*(oz|ounce|lb|pound|g|kg|ml|l|fl oz|fl\. oz\.)\b', text, flags=re.I)
    if qty_match:
        features['numeric_quantity'] = float(qty_match.group(1))
        features['quantity_unit'] = qty_match.group(2).lower().replace('.', '')
    else:
        features['numeric_quantity'] = np.nan
        features['quantity_unit'] = 'unknown'

    # Brand (more robustly captures brands with numbers or multiple words)
    brand_match = re.search(r"Item Name:\s*([A-Z0-9][A-Za-z0-9' -]{1,30})\b", text)
    features['brand'] = brand_match.group(1).strip() if brand_match else 'Unknown'

    attribute_keywords = [
        'premium', 'organic', 'gourmet', 'heavy-duty', 'professional', 'industrial',
        'handmade', 'natural', 'usda', 'gluten-free', 'non-gmo', 'eco-friendly',
        'wireless', 'bluetooth', 'smart', 'hd', '4k', 'waterproof'
    ]
    for keyword in attribute_keywords:
        features[f'attr_{keyword.replace("-", "_")}'] = 1 if keyword in lower_text else 0

    materials = ['wood', 'steel', 'stainless steel', 'leather', 'cotton', 'plastic', 'ceramic', 'glass', 'aluminum', 'copper']
    for material in materials:
        features[f'mat_{material.replace(" ", "_")}'] = 1 if material in lower_text else 0

    colors = ['red', 'blue', 'black', 'white', 'green', 'yellow', 'silver', 'gold', 'brown', 'purple', 'orange']
    for color in colors:
        features[f'color_{color}'] = 1 if color in lower_text else 0

    features['text_length'] = len(text)
    features['word_count'] = len(text.split())
    features['bullet_point_count'] = text.count("Bullet Point")
    
    desc_match = re.search(r"Product Description:\s*(.+)", text, flags=re.S | re.I)
    features['description_length'] = len(desc_match.group(1).strip()) if desc_match else 0
    
    return features

print("2. Applying feature engineering...")
train_features_df = train_df['catalog_content'].progress_apply(extract_features).apply(pd.Series)
test_features_df = test_df['catalog_content'].progress_apply(extract_features).apply(pd.Series)

2. Applying feature engineering...


  0%|          | 0/74972 [00:00<?, ?it/s]

  0%|          | 0/75000 [00:00<?, ?it/s]

In [ ]:
print("3. Vectorizing engineered features...")

numerical_cols = [
    'pack_count', 'numeric_quantity', 'text_length', 
    'word_count', 'bullet_point_count', 'description_length'
] + [col for col in train_features_df.columns if col.startswith(('attr_', 'mat_', 'color_'))]

# High-cardinality categorical columns (many unique values)
high_card_categorical_cols = ['brand']

# Low-cardinality categorical columns (few unique values)
low_card_categorical_cols = ['quantity_unit']

# Pipeline for numerical features:
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipelinea for high-cardinality text features (e.g., Brand):
brand_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('flatten', FunctionTransformer(lambda x : x.ravel(), validate = False)),
    ('tfidf', TfidfVectorizer(max_features=500, token_pattern=r'\b[a-zA-Z0-9-]+\b'))
])

# --- Use ColumnTransformer to apply pipelines to the correct columns ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_cols),
        ('brand_tfidf', brand_pipeline
        , ['brand']),
        ('unit_onehot', OneHotEncoder(handle_unknown='ignore'), low_card_categorical_cols)
    ],
    remainder='drop'
)

print("   - Fitting preprocessor and transforming data...")
train_engineered_feats = preprocessor.fit_transform(train_features_df)
test_engineered_feats = preprocessor.transform(test_features_df)

if hasattr(train_engineered_feats, "toarray"):
    train_engineered_feats = train_engineered_feats.toarray()
    test_engineered_feats = test_engineered_feats.toarray()

print(f"\nSuccessfully created engineered feature matrices.")
print(f"Engineered training feature shape: {train_engineered_feats.shape}")
print(f"Engineered test feature shape: {test_engineered_feats.shape}")


3. Vectorizing engineered features...
   - Fitting preprocessor and transforming data...

Successfully created engineered feature matrices.
Engineered training feature shape: (74972, 555)
Engineered test feature shape: (75000, 555)


In [ ]:
print("\n4. Combining all features into final matrices...")
X = np.hstack([train_engineered_feats, train_text_embeds, train_image_embeds])
X_test = np.hstack([test_engineered_feats, test_text_embeds, test_image_embeds])
y = train_df['price'].values
y_log = np.log1p(y) # Use log-transform for training

print(f"Final training matrix shape: {X.shape}")
print(f"Final test matrix shape: {X_test.shape}")
print(f"y_log shape: {y_log.shape}")

del train_engineered_feats, train_text_embeds, train_image_embeds
del test_engineered_feats, test_text_embeds, test_image_embeds
gc.collect()


4. Combining all features into final matrices...
Final training matrix shape: (74972, 4011)
Final test matrix shape: (75000, 4011)
y_log shape: (74972,)


46

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

 ## Version 1

In [ ]:
print("\n5. Starting Level 0 training...")

# Use StratifiedKFold on binned target for robust validation
skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.SEED)
train_df['price_bin'] = pd.cut(train_df['price'], bins=10, labels=False, include_lowest=True)

# Arrays to store OOF and test predictions for Level 0 models
oof_l0_lgbm = np.zeros(len(train_df))
oof_l0_ridge = np.zeros(len(train_df))
test_l0_lgbm = np.zeros(len(test_df))
test_l0_ridge = np.zeros(len(test_df))

# --- Define Level 0 models ---
lgbm_params = {'objective': 'regression_l1', 'metric': 'mae', 'n_estimators': 1000, 'learning_rate': 0.05, 'random_state': Config.SEED}
ridge_model = Ridge(random_state=Config.SEED, alpha=1.0)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, train_df['price_bin'])):
    print(f"  ===== FOLD {fold+1}/{Config.N_SPLITS} =====")
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y_log[train_idx], y_log[val_idx]
    
    # --- Train LightGBM ---
    print("    - Training LightGBM...")
    lgbm_model = lgb.LGBMRegressor(**lgbm_params)
    lgbm_model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_l0_lgbm[val_idx] = lgbm_model.predict(X_val_fold)
    test_l0_lgbm += lgbm_model.predict(X_test) / Config.N_SPLITS
    
    # --- Train Ridge ---
    print("    - Training Ridge Regressor...")
    ridge_model.fit(X_train_fold, y_train_fold)
    oof_l0_ridge[val_idx] = ridge_model.predict(X_val_fold)
    test_l0_ridge += ridge_model.predict(X_test) / Config.N_SPLITS
    
    del lgbm_model
    gc.collect()

# --- Create input for the next level (the meta-model) ---
X_meta_l1 = np.column_stack([oof_l0_lgbm, oof_l0_ridge])
X_meta_l1_test = np.column_stack([test_l0_lgbm, test_l0_ridge])

In [ ]:
print("\n4. Training Level 1 final blender (Lasso Regressor)...")

# Initialize and train the final blender on the full set of OOF predictions
final_blender = Lasso(random_state=Config.SEED, alpha=0.001)
final_blender.fit(X_meta_l1, y_log)

# --- Generate Final Predictions ---
final_preds_log = final_blender.predict(X_meta_l1_test)
final_preds = np.expm1(final_preds_log) # Inverse transform from log scale

# Also get OOF predictions from the final blender for validation
oof_final_preds_log = final_blender.predict(X_meta_l1)
oof_final_preds = np.expm1(oof_final_preds_log)

In [ ]:
final_oof_smape = smape(train_df['price'], oof_final_preds)
print(f"\nOverall Final Out-of-Fold SMAPE: {final_oof_smape:.4f}")

In [ ]:
submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': final_preds
})
submission_df['price'] = submission_df['price'].clip(lower=0.01) # Ensure no negative prices
submission_df.to_csv('submission.csv', index=False)

print("\nSubmission file 'submission.csv' created successfully.")
print(submission_df.head())

In [ ]:


submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': final_preds
})
submission_df['price'] = submission_df['price'].clip(lower=0.01) # Ensure no negative prices
submission_df.to_csv('submission.csv', index=False)

print("\nSubmission file 'submission.csv' created successfully.")
print(submission_df.head())

 ## Version 2

In [ ]:
# --- Model 1: XGBoost ---
xgb_params = {'objective': 'reg:squarederror', "early_stopping_rounds" : 50, 'eval_metric': 'rmse', 'seed': config.SEED, 'n_estimators': 1000, 'learning_rate': 0.05, 'tree_method': 'hist', "device" : "cuda"}

# --- Model 2: Artificial Neural Network (ANN) ---
def create_ann_model(input_shape):
    inp = layers.Input(shape=(input_shape,))
    x = layers.BatchNormalization()(inp)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(1, activation='relu')(x)
    model = keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=config.ANN_LEARNING_RATE), loss='mean_squared_error')
    return model

In [ ]:
print("\n5. Starting Level 0 training...")

# Use StratifiedKFold on binned target for robust validation
skf = StratifiedKFold(n_splits=config.N_SPLITS, shuffle=True, random_state=config.SEED)
price_bins = pd.cut(train_df['price'], bins=10, labels=False, include_lowest=True)
train_df['price_bin'] = np.where(price_bins.value_counts()[price_bins] <= 2, 2, price_bins)


# Arrays to store OOF and test predictions for Level 0 models
oof_l0_xgb = np.zeros(len(train_df))
oof_l0_ann = np.zeros(len(train_df))
test_l0_xgb = np.zeros(len(test_df))
test_l0_ann = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, train_df['price_bin'])):
    print(f"  ===== FOLD {fold+1}/{config.N_SPLITS} =====")
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y_log[train_idx], y_log[val_idx]
    
    # --- Train XGBoost ---
    print("    - Training XGBoost...")
    xgb_l0_model = xgb.XGBRegressor(**xgb_params)
    xgb_l0_model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], verbose=False)
    oof_l0_xgb[val_idx] = xgb_l0_model.predict(X_val_fold)
    test_l0_xgb += xgb_l0_model.predict(X_test) / config.N_SPLITS
    
    # --- Train ANN ---
    print("    - Training ANN...")
    ann_l0_model = create_ann_model(X.shape[1])
    early_stopper = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ann_l0_model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), 
                  epochs=config.ANN_EPOCHS, batch_size=config.ANN_BATCH_SIZE, 
                  callbacks=[early_stopper], verbose=0)
    oof_l0_ann[val_idx] = ann_l0_model.predict(X_val_fold).flatten()
    test_l0_ann += ann_l0_model.predict(X_test).flatten() / config.N_SPLITS
    
    del xgb_l0_model, ann_l0_model
    gc.collect()

# --- Create input for the next level ---
X_l1 = np.column_stack([oof_l0_xgb, oof_l0_ann])
X_l1_test = np.column_stack([test_l0_xgb, test_l0_ann])

In [ ]:
print("\n4. Starting Level 1 meta-model training...")

# Arrays for Level 1 predictions
oof_l1_gbtree = np.zeros(len(train_df))
oof_l1_linear = np.zeros(len(train_df))
test_l1_gbtree = np.zeros(len(test_df))
test_l1_linear = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_l1, train_df['price_bin'])):
    X_train_fold, X_val_fold = X_l1[train_idx], X_l1[val_idx]
    y_train_fold, y_val_fold = y_log[train_idx], y_log[val_idx]
    
    # --- Train Non-linear Model (LGBM gbtree) ---
    lgbm_l1_model = lgb.LGBMRegressor(random_state=config.SEED)
    lgbm_l1_model.fit(X_train_fold, y_train_fold)
    oof_l1_gbtree[val_idx] = lgbm_l1_model.predict(X_val_fold)
    
    # --- Train Linear Model (Ridge) ---
    ridge_l1_model = Ridge(random_state=config.SEED)
    ridge_l1_model.fit(X_train_fold, y_train_fold)
    oof_l1_linear[val_idx] = ridge_l1_model.predict(X_val_fold)

# Train Level 1 models on full OOF data to predict on test data
lgbm_l1_model.fit(X_l1, y_log)
test_l1_gbtree = lgbm_l1_model.predict(X_l1_test)
ridge_l1_model.fit(X_l1, y_log)
test_l1_linear = ridge_l1_model.predict(X_l1_test)

# --- Create input for the final level ---
X_meta = np.column_stack([oof_l1_gbtree, oof_l1_linear])
X_meta_test = np.column_stack([test_l1_gbtree, test_l1_linear])

In [ ]:
print("\n5. Training meta model...")

meta_model = Lasso(random_state=config.SEED)
meta_model.fit(X_meta, y_log)

# --- Generate Final Predictions ---
final_preds_log = meta_model.predict(X_meta_test)
final_preds = np.expm1(final_preds_log) # Inverse transform from log scale

# Also get OOF predictions from the meta model for validation
oof_final_preds_log = meta_model.predict(X_meta)
oof_final_preds = np.expm1(oof_final_preds_log)

In [ ]:
final_oof_smape = smape(train_df['price'], oof_final_preds)
print(f"\nOverall Final Out-of-Fold SMAPE: {final_oof_smape:.4f}")

In [ ]:

submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': final_preds
})
submission_df['price'] = submission_df['price'].clip(lower=0.01)
submission_df.to_csv('/kaggle/working/submission.csv', index=False)

print("\nSubmission file 'submission.csv' created successfully.")
print(submission_df.head())

 ## Version 3

In [ ]:
# --- Model 1: XGBoost ---
xgb_params = {'objective': 'reg:squarederror', "early_stopping_rounds" : 50, 'eval_metric': 'rmse', 'seed': config.SEED, 'n_estimators': 1000, 'learning_rate': 0.05, 'tree_method': 'hist', "device" : "cuda"}

# --- Model 2: Artificial Neural Network (ANN) ---
def create_ann_model(input_shape):
    inp = layers.Input(shape=(input_shape,))
    x = layers.BatchNormalization()(inp)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(1, activation='relu')(x)
    model = keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=config.ANN_LEARNING_RATE), loss='mean_squared_error')
    return model

# --- Model 3: Another diverse model (LightGBM for speed and different algorithm) ---
lgbm_params = {'objective': 'regression_l1', 'metric': 'mae', 'n_estimators': 1000, 'learning_rate': 0.05, 'random_state': Config.SEED, "device" : "gpu"}

In [ ]:
print("\n5. Starting Level 0 training...")

# Use StratifiedKFold on binned target for robust validation
skf = StratifiedKFold(n_splits=config.N_SPLITS, shuffle=True, random_state=config.SEED)
price_bins = pd.cut(train_df['price'], bins=10, labels=False, include_lowest=True)
train_df['price_bin'] = np.where(price_bins.value_counts()[price_bins] <= 2, 2, price_bins)


# Arrays to store OOF and test predictions for Level 0 models
oof_l0_xgb = np.zeros(len(train_df))
oof_l0_ann = np.zeros(len(train_df))
oof_l0_lgbm = np.zeros(len(train_df))
test_l0_xgb = np.zeros(len(test_df))
test_l0_ann = np.zeros(len(test_df))
test_l0_lgbm = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, train_df['price_bin'])):
    print(f"  ===== FOLD {fold+1}/{config.N_SPLITS} =====")
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    y_train_fold, y_val_fold = y_log[train_idx], y_log[val_idx]
    
    # --- Train XGBoost ---
    print("    - Training XGBoost...")
    xgb_l0_model = xgb.XGBRegressor(**xgb_params)
    xgb_l0_model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], verbose=False)
    oof_l0_xgb[val_idx] = xgb_l0_model.predict(X_val_fold)
    test_l0_xgb += xgb_l0_model.predict(X_test) / config.N_SPLITS
    
    # --- Train ANN ---
    print("    - Training ANN...")
    ann_l0_model = create_ann_model(X.shape[1])
    early_stopper = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ann_l0_model.fit(X_train_fold, y_train_fold, validation_data=(X_val_fold, y_val_fold), 
                  epochs=config.ANN_EPOCHS, batch_size=config.ANN_BATCH_SIZE, 
                  callbacks=[early_stopper], verbose=0)
    oof_l0_ann[val_idx] = ann_l0_model.predict(X_val_fold).flatten()
    test_l0_ann += ann_l0_model.predict(X_test).flatten() / config.N_SPLITS
    
    # --- Train LightGBM ---
    print("    - Training LightGBM...")
    lgbm_model = lgb.LGBMRegressor(**lgbm_params)
    lgbm_model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_l0_lgbm[val_idx] = lgbm_model.predict(X_val_fold)
    test_l0_lgbm += lgbm_model.predict(X_test) / Config.N_SPLITS
    
    del xgb_model, ann_model, lgbm_model
    gc.collect()


5. Starting Level 0 training...
  ===== FOLD 1/3 =====
    - Training XGBoost...
    - Training ANN...
781/781 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
2344/2344 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
    - Training LightGBM...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 907983
[LightGBM] [Info] Number of data points in the train set: 49981, number of used features: 4008
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 3464 dense feature groups (165.11 MB) transferred to GPU in 0.155032 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 2.708384


In [ ]:
X_l1 = np.column_stack([train_engineered_feats, oof_l0_xgb, oof_l0_ann, oof_l0_lgbm])
X_l1_test = np.column_stack([test_engineered_feats, test_l0_xgb, test_l0_ann, test_l0_lgbm])

scaler = StandardScaler()
X_l1 = scaler.fit_transform(X_l1)
X_l1_test = scaler.transform(X_l1_test)

In [ ]:
print("\n4. Starting Level 1 meta-model training...")

# Arrays for Level 1 predictions
oof_l1_gbtree = np.zeros(len(train_df))
oof_l1_linear = np.zeros(len(train_df))
test_l1_gbtree = np.zeros(len(test_df))
test_l1_linear = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_l1, train_df['price_bin'])):
    X_train_fold, X_val_fold = X_l1[train_idx], X_l1[val_idx]
    y_train_fold, y_val_fold = y_log[train_idx], y_log[val_idx]
    
    # --- Train Non-linear Model (LGBM gbtree) ---
    lgbm_l1_model = lgb.LGBMRegressor(random_state=config.SEED)
    lgbm_l1_model.fit(X_train_fold, y_train_fold)
    oof_l1_gbtree[val_idx] = lgbm_l1_model.predict(X_val_fold)
    
    # --- Train Linear Model (Ridge) ---
    ridge_l1_model = Ridge(random_state=config.SEED)
    ridge_l1_model.fit(X_train_fold, y_train_fold)
    oof_l1_linear[val_idx] = ridge_l1_model.predict(X_val_fold)

# Train Level 1 models on full OOF data to predict on test data
lgbm_l1_model.fit(X_l1, y_log)
test_l1_gbtree = lgbm_l1_model.predict(X_l1_test)
ridge_l1_model.fit(X_l1, y_log)
test_l1_linear = ridge_l1_model.predict(X_l1_test)

# --- Create input for the final level ---
X_meta = np.column_stack([oof_l1_gbtree, oof_l1_linear])
X_meta_test = np.column_stack([test_l1_gbtree, test_l1_linear])


4. Starting Level 1 meta-model training...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.093319 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 27213
[LightGBM] [Info] Number of data points in the train set: 49981, number of used features: 554
[LightGBM] [Info] Start training from score 2.738287
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.079759 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 27395
[LightGBM] [Info] Number of data points in the train set: 49981, number of used features: 554
[LightGBM] [Info] Start training from score 2.740802
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.079985 seconds.
You can set `force_row_wise=true` to

In [ ]:
print("\n5. Training meta model...")

meta_model = Lasso(random_state=config.SEED)
meta_model.fit(X_meta, y_log)

# --- Generate Final Predictions ---
final_preds_log = meta_model.predict(X_meta_test)
final_preds = np.expm1(final_preds_log) # Inverse transform from log scale

# Also get OOF predictions from the meta model for validation
oof_final_preds_log = meta_model.predict(X_meta)
oof_final_preds = np.expm1(oof_final_preds_log)


5. Training meta model...


In [ ]:
final_oof_smape = smape(train_df['price'], oof_final_preds)
print(f"\nOverall Final Out-of-Fold SMAPE: {final_oof_smape:.4f}")


Overall Final Out-of-Fold SMAPE: 72.6973


In [ ]:

submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': final_preds
})
submission_df['price'] = submission_df['price'].clip(lower=0.01)
submission_df.to_csv('/kaggle/working/submission.csv', index=False)

print("\nSubmission file 'submission.csv' created successfully.")
print(submission_df.head())


Submission file 'submission.csv' created successfully.
   sample_id      price
0     100179  14.474143
1     245611  14.474143
2     146263  14.474143
3      95658  14.474143
4      36806  14.474143
